In [7]:
# Comparing of list of Annotated View records with records with metadata from Excel
"""
takes a DataFrame read from Excel and returns it sorted exactly as you described:
- group by userId
- within each group: sort by (S+V) descending
- tie-break: ann_nz_len ascending (smaller first)
- keeps full rows together (stable sort)

https://chatgpt.com/c/6981cb46-2a20-8395-8ba7-fabc820ff905

"""
from __future__ import annotations

import json
from pathlib import Path
from typing import Dict, List, Optional, Tuple, TypedDict, Any, Set
from collections import Counter
import pandas as pd
import sys
import numpy as np


# funkcija skaito Excel su įrašų sąrašu ir meta duomenimis 
def read_filenames_from_excel(xlsx_path: Path) -> tuple[list[str], pd.DataFrame]:
    """Skaito Excel, grąžina sąrašą `.npy` failų ir visą DF (meta duomenims paimti)."""
    print("\nSkaitomas Excel: %s", xlsx_path)
    df = pd.read_excel(xlsx_path, dtype=str)  # saugu tolimesnėms konversijoms
    if "filename" not in df.columns or "tag" not in df.columns:
        raise ValueError("Excel must contain columns: 'filename' and 'tag'")

    # filtered = df[df["tag"] != "9999"]
    filtered = df
    names = []
    for s in filtered["filename"].dropna():
        name = str(s).strip()
        if not name.endswith(".npy"):
            name += ".npy"
        names.append(name)

    print("Įrašų sąraše:", len(names))
    return names, df

def _read_json_data(json_path: Path) -> Optional[Dict[str, Any]]:
    """
    Reads JSON file and returns parsed dict.
    Returns None if file does not exist, JSON is invalid, or top-level is not a dict.
    """
    if not json_path.exists():
        return None

    try:
        with open(json_path, "r", encoding="utf-8", errors="ignore") as f:
            data = json.load(f)
    except (json.JSONDecodeError, OSError):
        return None

    if not isinstance(data, dict):
        return None

    return data

from pathlib import Path
from typing import Any, Dict, Optional, Set, List
import json
import pandas as pd

# assumes these already exist in your code:
# _read_json_data(json_path: Path) -> Optional[Dict[str, Any]]
# _extract_raw_flags_set(data: Dict[str, Any]) -> Optional[Set[str]]

DEFAULT_KNOWN_FLAGS = [
    "PSEUDO_ANNOTATED",
    "FOR_EXTERNAL_ANNOTATING",
    "EXTERNALLY_ANNOTATED",
    "FULLY_ANNOTATED_PROFESSIONALLY",
]

def flags_str_for_filename(
    fname: str,
    rec_dir: Path,
    known_flags: Optional[List[str]] = None,
) -> str:
    """
    Reads <rec_dir>/<fname>.json, extracts flags, and returns a comma-separated string
    of known_flags that are present. Returns "" if missing/invalid.
    """
    known_flags = known_flags or DEFAULT_KNOWN_FLAGS

    json_path = (rec_dir / fname).with_suffix(".json")

    data = _read_json_data(json_path)
    if data is None:
        return ""  # or "MISSING/INVALID"

    raw_flags_set = _extract_raw_flags_set(data)
    if raw_flags_set is None:
        return ""  # invalid flags structure

    return ",".join(sorted(raw_flags_set))


def _extract_raw_flags_set(data: Dict[str, Any]) -> Optional[Set[str]]:
    """
    Extracts flags from JSON data and returns them as a set of strings.
    Returns None if 'flags' exists but is not a list (invalid structure).
    """
    raw_flags = data.get("flags")

    if raw_flags is None:
        raw_flags_list: List[str] = []
    elif isinstance(raw_flags, list):
        raw_flags_list = [str(x) for x in raw_flags]
    else:
        return None  # invalid structure

    return set(raw_flags_list)


def sort_by_user_sv_annot_len(
    df: pd.DataFrame,
    user_col: str = "userId",
    s_col: str = "S",
    v_col: str = "V",
    ann_len_col: str = "ann_nz_len",
    user_order: str = "asc",          # "asc" or "preserve"
    na_ann_len_last: bool = True,     # put missing ann_nz_len at bottom within ties
    add_group_no: bool = True,        # add sequential group number column
    add_group_first: bool = False,    # add boolean flag for first row in each group
    group_no_col: str = "group_no",
    group_first_col: str = "group_first",
) -> pd.DataFrame:
    """
    Sort rows as:
      1) group by userId
      2) inside each userId: sort by (S+V) descending
      3) tie-break: smaller ann_nz_len first

    Extras:
      - add sequential group number (1..K) as first column (optional)
      - add boolean flag for first row in each group (optional)

    user_order:
      - "asc": sort userId groups lexicographically
      - "preserve": keep userId group order as they first appear in the original df
    """
    missing = [c for c in (user_col, s_col, v_col, ann_len_col) if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")

    d = df.copy()

    # Numeric conversion; missing/non-numeric -> 0 for S/V
    s_num = pd.to_numeric(d[s_col], errors="coerce").fillna(0.0)
    v_num = pd.to_numeric(d[v_col], errors="coerce").fillna(0.0)
    sv_sum = s_num + v_num

    # ann_nz_len numeric; missing -> +inf if requested (so it sorts last in tie-break)
    ann_len = pd.to_numeric(d[ann_len_col], errors="coerce")
    ann_len_sort = ann_len.fillna(np.inf) if na_ann_len_last else ann_len

    # userId key
    user_raw = d[user_col].astype(str).fillna("")

    if user_order == "preserve":
        # group order = first appearance in original df
        user_key, _ = pd.factorize(user_raw, sort=False)  # 0..K-1
    elif user_order == "asc":
        user_key = user_raw
    else:
        raise ValueError("user_order must be 'asc' or 'preserve'")

    # Temporary sort columns
    d["_user_key__tmp"] = user_key
    d["_sv_sum__tmp"] = sv_sum
    d["_ann_len__tmp"] = ann_len_sort

    # Stable sort
    d = d.sort_values(
        by=["_user_key__tmp", "_sv_sum__tmp", "_ann_len__tmp"],
        ascending=[True, False, True],
        kind="mergesort",
    ).drop(columns=["_user_key__tmp", "_sv_sum__tmp", "_ann_len__tmp"])

    # Identify first row of each (now-contiguous) userId group in the sorted output
    is_first = d[user_col].astype(str).fillna("").ne(d[user_col].astype(str).fillna("").shift(1))

    # Sequential group numbers in output order
    group_no = is_first.cumsum().astype(int)

    if add_group_first:
        # Put it next to group_no (or as first col if group_no not added)
        insert_at = 0
        d.insert(insert_at, group_first_col, is_first.values)

    if add_group_no:
        d.insert(0, group_no_col, group_no.values)

    return d

from openpyxl.styles import PatternFill


def write_excel_with_group_first_highlight(
    df: pd.DataFrame,
    out_path: str | Path,
    group_first_col: str = "group_first",
    sheet_name: str = "Sheet1",
    fill_hex: str = "FFF2F2F2",   # very light gray (Excel fill has no real transparency)
    freeze_header: bool = True,
) -> Path:
    """
    Write df to an .xlsx file and highlight full rows where df[group_first_col] is True.

    Notes:
    - Excel doesn't support true transparency in cell fills; use a light tint instead.
    - Assumes df contains a boolean-like column `group_first_col` (True for first row per group).
    """
    out_path = Path(out_path)

    if group_first_col not in df.columns:
        raise KeyError(
            f"Column '{group_first_col}' not found. "
            f"Call sort_by_user_sv_annot_len(..., add_group_first=True) first."
        )

    # Ensure boolean mask
    mask = df[group_first_col].fillna(False).astype(bool).values

    highlight_fill = PatternFill(fill_type="solid", fgColor=fill_hex)

    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=False)
        ws = writer.book[sheet_name]

        # Highlight rows (Excel row index: header=1, first data row=2)
        max_col = ws.max_column
        for i, is_first in enumerate(mask):
            if not is_first:
                continue
            excel_row = i + 2
            for col in range(1, max_col + 1):
                ws.cell(row=excel_row, column=col).fill = highlight_fill

        if freeze_header:
            ws.freeze_panes = "A2"

    return out_path


# csv_path = Path("AnnotatedViewList_2026_01_22.csv")  # or full path
# df = pd.read_csv(csv_path)

# If you want a list of timestamps as strings (already like xxxxxxx.xxx)
# csv_names = df["timestamp"].astype(str).tolist()

# Filenames with a suffix you choose
# filenames_npy  = [f"{t}.npy" for t in timestamps]
# filenames_json = [f"{t}.json" for t in timestamps]
# filenames_dat  = [f"{t}.dat" for t in timestamps]

# print(csv_names)

# === Išoriniai moduliai (lygiagretus aplankas) =================================
PARALLEL_PATH = Path().resolve().parent / "../SUPL_FUNCTIONS"
sys.path.append(str(PARALLEL_PATH))

from project_util import find_project_root_by_name


PROJECT_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PROJECT_ROOT = find_project_root_by_name(target="PROJECT_TRAIN_UNET", start=PROJECT_DIR)
print("\nPROJECT ROOT DIR:", PROJECT_ROOT)
print("PROJECT DIR:", PROJECT_DIR)

LIST_DIR = PROJECT_ROOT / '1_PREPARE_TRAIN_UNET_DATA'/ 'ecg_zive_npy_for_preparing'
REC_DIR = PROJECT_ROOT / "DATA_ORIG/ecg_zive_npy"
# EXCEL_NAME = "visi_zive_irasai_atrankai_test.xlsx"
EXCEL_NAME = "visi_zive_irasai_atrankai._extended.xlsx"

# \\wsl.localhost\Ubuntu\home\kesju\DI\2025_ZIVEO\PROJECT_TRAIN_UNET\1_PREPARE_TRAIN_UNET_DATA\ecg_zive_npy_for_preparing\visi_zive_irasai_atrankai._extended.xlsx

print("\nLIST_DIR:", LIST_DIR)
print("REC_DIR:", REC_DIR)
print("EXCEL_NAME:", EXCEL_NAME)

# file_names, df_meta = read_filenames_from_excel(LIST_DIR / EXCEL_NAME)
# print(file_names)

# df = pd.read_excel(LIST_DIR / EXCEL_NAME)
# # df_sorted = sort_by_user_sv_annot_len(df, user_order="asc")
# # or keep original userId group order:
# df_sorted = sort_by_user_sv_annot_len(df, user_order="preserve")

# out_xlsx_path = Path(f"sorted_{EXCEL_NAME}")
# df_sorted.to_excel(out_xlsx_path, index=False)
# print(f"Wrote: {out_xlsx_path.resolve()}")

df = pd.read_excel(LIST_DIR / EXCEL_NAME)

df_sorted = sort_by_user_sv_annot_len(
    df,
    user_order="preserve",
    add_group_no=True,
    add_group_first=True,
)

"""
Light blue
Very light blue: FFEAF2FF
Light sky blue: FFD9ECFF
Light teal-blue: FFD6F5FF

Light yellow
Very light yellow: FFFFF7CC
Soft yellow: FFFFF2B3
Pale lemon: FFFFF1A8

Light green
Very light green: FFE6F6E6
Soft mint: FFDFF5E1
Pale green: FFDCFCE7
"""

write_excel_with_group_first_highlight(
    df_sorted,
    "visi_zive_irasai_atrankai._extended_sorted_grouped_highlighted.xlsx",
    fill_hex="FFFFF2B3",  # soft yellow
)



# matches = matches.copy()
# matches["flags"] = matches["filename"].astype(str).apply(flags_from_filename)


# cols_to_show = [c for c in ["filename", "recordingId", "userId", "tag", "basename", "flags"] if c in matches.columns]
# print("\n=== MATCHES (same basename) ===")
# print(matches[cols_to_show].to_string(index=False))

# Surandami unikalūs userId ir jų pasikartojimai
# user_counts_df = (
#     matches["userId"]
#     .dropna()
#     .astype(str)
#     .value_counts()
#     .rename_axis("userId")
#     .reset_index(name="count")
# )
# print(user_counts_df.head())
# print("\nSurandami unikalūs userId ir jų pasikartojimai:")
# print(user_counts_df.to_string(index=False))

# 6144c682bd0cc5acb727536 inmed20@zive.io


# print matching cases (choose the columns you want to see)
# cols_to_show = [c for c in ["filename", "recordingId", "tag", "basename"] if c in matches.columns]
# print("\n=== MATCHES (same basename) ===")
# print(matches[cols_to_show].to_string(index=False))

# If you also want to print CSV entries not found in Excel and Excel entries not in CSV, tell me and I’ll add the two “missing” lists (it’s ~10 extra lines).




PROJECT ROOT DIR: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET
PROJECT DIR: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/1_PREPARE_TRAIN_UNET_DATA/ScriptsForAnalysis

LIST_DIR: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/1_PREPARE_TRAIN_UNET_DATA/ecg_zive_npy_for_preparing
REC_DIR: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/DATA_ORIG/ecg_zive_npy
EXCEL_NAME: visi_zive_irasai_atrankai._extended.xlsx


PosixPath('visi_zive_irasai_atrankai._extended_sorted_grouped_highlighted.xlsx')